In [ ]:
import importlib
import gc
import math

import numpy as np
import pandas as pd
import data_cleaning as dc

from sklearn.neighbors import BallTree
from transit_features import get_nearest_subway

gc.collect()
importlib.reload(dc)

<module 'data_cleaning' from '/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_Final/src/data_cleaning.py'>

In [2]:
prop_values_df = dc.batch_load_from_csv(path="../data/clean_data/val/cleaned_property_part", num_files=8)
bus_df = pd.read_csv("../data/clean_data/bus/clean_bus_df.csv")
sub_df = pd.read_csv("../data/clean_data/sub/cleaned_subway_entrances.csv")

/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_Final/src/data_cleaning.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(fp))
/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_Final/src/data_cleaning.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(fp))
/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_Final/src/data_cleaning.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(fp))
/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_Final/src/data_cleaning.py:40: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(fp))
/Users/zach/Desktop/Grad School Courses/Urban Computing/UrbanComputing_F

Concatting All...


In [ ]:
print(sub_df['Routes'])

def get_nearest_subway_entrance

0         R
1         R
2       N,R
3       N,R
4       N,R
       ... 
1848    2,5
1849    2,5
1850    2,5
1851    2,5
1852    2,5
Name: Routes, Length: 1853, dtype: object


In [52]:
sub_df.rename({'Entrance Longitude': 'Longitude', 'Entrance Latitude': 'Latitude'}, axis=1, inplace=True)

bus_df['subway'] = 0
sub_df['subway'] = 1

bus_locs = bus_df[['Latitude', 'Longitude', 'subway']]
sub_locs = sub_df[['Latitude', 'Longitude', 'subway']]

merged_transit = pd.concat([bus_locs, sub_locs], ignore_index=True)

We love Geeks4Geeks
https://www.geeksforgeeks.org/machine-learning/ball-tree-and-kd-tree-algorithms/#

In [102]:
bus_coords = np.deg2rad(bus_locs[['Latitude', 'Longitude']].to_numpy())
sub_coords = np.deg2rad(sub_locs[['Latitude', 'Longitude']].to_numpy())

bus_tree = BallTree(bus_coords, metric='haversine')
sub_tree = BallTree(sub_coords, metric='haversine')

query = prop_values_df.head(1)[['Latitude', 'Longitude']].to_numpy()
query = np.deg2rad(query)

near_bus_dist, near_bus_idx = bus_tree.query(query, k=3)
near_sub_dist, near_sub_idx = sub_tree.query(query, k=3)

In [94]:
near_bus_dist = near_bus_dist * 6371000  # convert to meters
near_sub_dist = near_sub_dist * 6371000  # convert to meters

In [96]:
print(f"Data for Location: \n\tLatitude: {prop_values_df.iloc[0]['Latitude']} \n\tLongitude: {prop_values_df.iloc[0]['Longitude']}")
print(f"Nearest Bus Stop: \n\tID: {near_bus_idx[0]} \n\tDistance: {near_bus_dist[0]} meters")
print(f"Nearest Subway Entrance: \n\tID: {near_sub_idx[0]} \n\tDistance: {near_sub_dist[0]} meters")

Data for Location: 
	Latitude: 40.708248 
	Longitude: -73.965936
Nearest Bus Stop: 
	ID: [183 760 759] 
	Distance: [349.30256005 375.82614796 452.12348579] meters
Nearest Subway Entrance: 
	ID: [255 258 256] 
	Distance: [592.74714559 602.30178723 658.70021133] meters


In [ ]:
meters_radius = 500
radians_radius = meters_radius / 6371000  # Earth's radius in meters

k_nearest_bus = bus_tree.query_radius(query, radians_radius, count_only=True)
k_nearest_sub = sub_tree.query_radius(query, radians_radius, count_only=True)

In [121]:
print(f"Number of Bus Stops within {meters_radius} meters: {k_nearest_bus[0]}")
print(f"Number of Subway Entrances within {meters_radius} meters: {k_nearest_sub[0]}")

Number of Bus Stops within 350 meters: 1
Number of Subway Entrances within 350 meters: 0
